# Video Segmentation with SAM3 using Bounding Box Prompts

This notebook demonstrates how to use SAM3 for interactive video segmentation using bounding box prompts. It covers:

- **Bounding box prompts**: Using rectangular regions to segment and track objects
- **Dense tracking**: Propagating segmentation masks throughout the entire video
- **Refinement**: Adjusting segmentation with multiple bounding boxes

In [ ]:
import os
import sam3
import torch

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

# use all available GPUs on the machine
gpus_to_use = range(torch.cuda.device_count())
# # use only a single GPU
# gpus_to_use = [torch.cuda.current_device()]

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

## Utilities

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
    draw_box_on_image,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12

def propagate_in_video(predictor, session_id):
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]
    return outputs_per_frame

def abs_to_rel_coords(coords, IMG_WIDTH, IMG_HEIGHT, coord_type="point"):
    """Convert absolute coordinates to relative coordinates (0-1 range)
    
    Args:
        coords: List of coordinates
        coord_type: 'point' for [x, y] or 'box' for [x, y, w, h]
    """
    if coord_type == "point":
        return [[x / IMG_WIDTH, y / IMG_HEIGHT] for x, y in coords]
    elif coord_type == "box":
        return [
            [x / IMG_WIDTH, y / IMG_HEIGHT, w / IMG_WIDTH, h / IMG_HEIGHT]
            for x, y, w, h in coords
        ]
    else:
        raise ValueError(f"Unknown coord_type: {coord_type}")

## Load Video

In [ ]:
# Load video frames
video_path = f"{sam3_root}/assets/videos/0001"

video_frames_for_vis = glob.glob(os.path.join(video_path, "*.jpg"))
video_frames_for_vis.sort(
    key=lambda p: int(os.path.splitext(os.path.basename(p))[0])
)
print(f"Loaded {len(video_frames_for_vis)} frames")

## Start Inference Session

In [ ]:
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
session_id = response["session_id"]
print(f"Started session: {session_id}")

## Segment with Bounding Box Prompt

Let's add a bounding box prompt to segment an object in frame 0, then propagate the segmentation throughout the video.

In [ ]:
# Get frame dimensions for coordinate conversion
sample_img = Image.fromarray(load_frame(video_frames_for_vis[0]))
IMG_WIDTH, IMG_HEIGHT = sample_img.size
print(f"Video dimensions: {IMG_WIDTH}x{IMG_HEIGHT}")

In [ ]:
# Define a bounding box in absolute coordinates (x, y, width, height)
# This box will segment the person in the foreground
box_abs_xywh = [480.0, 290.0, 110.0, 360.0]  # (x, y, w, h)

# Convert to relative coordinates (0-1 range)
box_rel_xywh = abs_to_rel_coords([box_abs_xywh], IMG_WIDTH, IMG_HEIGHT, coord_type="box")[0]
print(f"Absolute box: {box_abs_xywh}")
print(f"Relative box: {box_rel_xywh}")

In [ ]:
# Add bounding box prompt on frame 0
frame_idx = 0
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        bounding_boxes=[box_rel_xywh],
        bounding_box_labels=[1],  # 1 = positive label
    )
)
out = response["outputs"]

# Visualize the segmentation result
plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM3 Bounding Box Segmentation"],
    figsize=(8, 6),
)

## Propagate Segmentation Through Video

In [ ]:
# Propagate the segmentation from frame 0 to all frames
outputs_per_frame = propagate_in_video(predictor, session_id)
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

print(f"Propagation complete. Segmentation results for {len(outputs_per_frame)} frames.")

In [ ]:
# Visualize results every 60 frames
vis_frame_stride = 60
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM3 Dense Tracking Results"],
        figsize=(8, 6),
    )

## Multiple Bounding Boxes

You can also add multiple bounding boxes to segment multiple objects in a single frame.

In [ ]:
# Reset the session to try multiple boxes
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

# Define two bounding boxes
boxes_abs = [
    [480.0, 290.0, 110.0, 360.0],  # Person in foreground
    [700.0, 200.0, 150.0, 300.0],  # Another region
]

# Convert to relative coordinates
boxes_rel = abs_to_rel_coords(boxes_abs, IMG_WIDTH, IMG_HEIGHT, coord_type="box")

frame_idx = 0

# Add bounding boxes one at a time (visual prompts only support one box as initial prompt)
for i, box_rel in enumerate(boxes_rel):
    response = predictor.handle_request(
        request=dict(
            type="add_prompt",
            session_id=session_id,
            frame_index=frame_idx,
            bounding_boxes=[box_rel],  # Pass one box at a time
            bounding_box_labels=[1],   # Positive label
            clear_old_boxes=(i == 0),  # Only clear on first iteration
        )
    )

out = response["outputs"]

plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM3 Multiple Bounding Boxes"],
    figsize=(8, 6),
)

In [ ]:
# Propagate with multiple boxes
outputs_per_frame = propagate_in_video(predictor, session_id)
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

# Visualize every 60 frames
vis_frame_stride = 60
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM3 Multiple Objects Tracking"],
        figsize=(8, 6),
    )

## Clean Up

In [ ]:
# Close the session to free up GPU memory
_ = predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

# Shutdown the predictor
predictor.shutdown()